# KMeans en mini-batch : comment ça marche vraiment ?

[Lien du post MonShotData](https://www.monshotdata.com/p/kmeans-mini-batch-fonctionnement)

In [1]:
import time
import numpy as np
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.metrics import pairwise_distances_argmin

## Générer les données

On utilise des clusters qui se chevauchent un peu, pour que la comparaison soit réaliste.

In [2]:
k = 8

X, _ = make_blobs(n_samples=200_000, n_features=20, centers=k, cluster_std=4.0, random_state=42)

X.shape

(200000, 20)

On fixe les centroïdes initiaux, pour que toutes les méthodes partent exactement du même point.

In [3]:
rng = np.random.default_rng(0)

init = X[rng.choice(len(X), k, replace=False)].copy()

## KMeans classique (référence)

In [4]:
kmeans = KMeans(n_clusters=k, init=init, n_init=1, algorithm="lloyd", tol=0).fit(X)

print(f"Itérations : {kmeans.n_iter_}")
print(f"Inertie    : {kmeans.inertia_:,.2f}")

Itérations : 184
Inertie    : 74,918,637.38


## KMeans par chunks : vecteur-somme et count

C'est la solution du shot : on charge les données chunk par chunk, et on garde seulement un **vecteur-somme** et un **count** par centroïde.

À la fin d'un passage complet sur les données, on divise la somme par le count, et on remet les deux à zéro avant le passage suivant.

In [5]:
def kmeans_chunks(X, init, chunk_size=10_000, max_iter=300):
    centers = init.copy()

    for n_iter in range(1, max_iter + 1):
        sums = np.zeros_like(centers)
        counts = np.zeros(len(centers))

        for start in range(0, len(X), chunk_size):
            chunk = X[start:start + chunk_size]

            labels = pairwise_distances_argmin(chunk, centers)
            np.add.at(sums, labels, chunk)
            counts += np.bincount(labels, minlength=len(centers))

        new_centers = np.where(counts[:, None] > 0, sums / np.maximum(counts, 1)[:, None], centers)

        if np.allclose(new_centers, centers, rtol=0, atol=1e-12):
            centers = new_centers
            break
        centers = new_centers

    return centers, n_iter

In [6]:
centers_chunks, n_iter_chunks = kmeans_chunks(X, init)

print(f"Itérations : {n_iter_chunks}")

Itérations : 184


Les centroïdes sont-ils identiques à ceux du KMeans classique ?

In [7]:
print("Mêmes centroïdes :", np.allclose(centers_chunks, kmeans.cluster_centers_, atol=1e-6))
print("Écart maximum    :", np.abs(centers_chunks - kmeans.cluster_centers_).max())

Mêmes centroïdes : True
Écart maximum    : 7.460698725481052e-14


Oui, aux erreurs d'arrondi près. Et la qualité du clustering (l'inertie) est la même :

In [8]:
def inertia(X, centers, chunk_size=20_000):
    total = 0.0

    for start in range(0, len(X), chunk_size):
        chunk = X[start:start + chunk_size]
        labels = pairwise_distances_argmin(chunk, centers)
        total += ((chunk - centers[labels]) ** 2).sum()

    return total

print(f"KMeans classique : {kmeans.inertia_:,.2f}")
print(f"KMeans par chunks : {inertia(X, centers_chunks):,.2f}")

KMeans classique : 74,918,637.38
KMeans par chunks : 74,918,637.38


## MiniBatchKMeans de Sklearn

Cette fois, Sklearn déplace les centroïdes après **chaque mini-batch**, sans attendre la fin d'un passage complet.

In [9]:
mini_batch = MiniBatchKMeans(n_clusters=k, init=init, n_init=1, batch_size=1024, random_state=0).fit(X)

print(f"Mini-batches traités : {mini_batch.n_steps_}")
print(f"Inertie              : {mini_batch.inertia_:,.2f}")

Mini-batches traités : 444
Inertie              : 74,939,157.08


In [10]:
ecart = (mini_batch.inertia_ - kmeans.inertia_) / kmeans.inertia_

print(f"Écart d'inertie avec le KMeans classique : {ecart:+.3%}")

Écart d'inertie avec le KMeans classique : +0.027%


Le résultat n'est **pas exactement le même** : `MiniBatchKMeans` est une approximation du KMeans classique.

L'écart est ici très faible, mais il existe.

Côté temps, tout dépend de la taille des données, du nombre de clusters et de ta machine : sur un dataset de cette taille, le mini-batch n'a pas forcément d'avantage. Voici une mesure (elle varie d'une machine à l'autre) :

In [11]:
start = time.perf_counter()
KMeans(n_clusters=k, init=init, n_init=1).fit(X)
print(f"KMeans          : {time.perf_counter() - start:.2f} s")

start = time.perf_counter()
MiniBatchKMeans(n_clusters=k, init=init, n_init=1, batch_size=1024, random_state=0).fit(X)
print(f"MiniBatchKMeans : {time.perf_counter() - start:.2f} s")

KMeans          : 0.20 s

MiniBatchKMeans : 0.88 s


## Ce que fait vraiment Sklearn : une moyenne glissante

Chaque centroïde garde un **count cumulé, jamais remis à zéro**. Après chaque mini-batch, sa position devient :

```
nouveau_centre = (ancien_centre × ancien_count + somme_du_batch) / (ancien_count + count_du_batch)
```

C'est le même truc somme/count que plus haut. Seule la fréquence de mise à jour change.

Vérifions en recodant cette formule à la main, et en la comparant à `partial_fit` de Sklearn. On désactive la réaffectation des centroïdes trop peu utilisés (`reassignment_ratio=0`) pour isoler la moyenne glissante.

In [12]:
batch_size = 1000

centers = init.copy()
counts = np.zeros(k)

sklearn_model = MiniBatchKMeans(n_clusters=k, init=init, n_init=1, reassignment_ratio=0, random_state=0)

for start in range(0, len(X), batch_size):
    batch = X[start:start + batch_size]

    # Notre moyenne glissante
    labels = pairwise_distances_argmin(batch, centers)

    for j in range(k):
        batch_points = batch[labels == j]
        batch_count = len(batch_points)

        if batch_count > 0:
            centers[j] = (centers[j] * counts[j] + batch_points.sum(axis=0)) / (counts[j] + batch_count)
            counts[j] += batch_count

    # Sklearn
    sklearn_model.partial_fit(batch)

In [13]:
print("Même résultat que Sklearn :", np.allclose(centers, sklearn_model.cluster_centers_, atol=1e-6))
print("Écart maximum             :", np.abs(centers - sklearn_model.cluster_centers_).max())

Même résultat que Sklearn : True
Écart maximum             : 7.993605777301127e-14
